# Week 6 loss–metric alignment diagnostic

Attach the private dataset containing `chronopde.h5`, enable Internet, and select one T4 GPU. This notebook trains fresh CT-FFT and DCT models on the fixed balanced batch with the diagnostic-only full-field relative objective. It preserves the original gate and does not launch later phases.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import traceback
import uuid
from pathlib import Path

from IPython.display import FileLink, display

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
IMPLEMENTATION_COMMIT = '107bbef5560caa1862ac04012320ace76216331a'
REPOSITORY_URL = 'https://github.com/madhavkapoor13/ChronoPDE.git'
EXPECTED_DATA_SIZE = 2_389_261_328
EXPECTED_DATA_SHA256 = (
    '907aa0d79e604e68ce2d4f5cccfd93ffc64eb68c473caf3edae4be438472caec'
)
RUN_ID = uuid.uuid4().hex[:8]
REPOSITORY = Path(f'/kaggle/working/Chrono_pde_alignment_{RUN_ID}')
SOURCE_OUTPUT = REPOSITORY / 'artifacts/diagnostics/week6/loss_alignment'
STATE_ROOT = Path(f'/kaggle/working/chronopde_alignment_state_{RUN_ID}')
PACKAGE_BASE = Path('/kaggle/working/chronopde_week6_loss_alignment')


In [ ]:
failure = None
return_code = None
STATE_ROOT.mkdir(parents=True, exist_ok=False)
try:
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError('Enable a T4 GPU in Kaggle notebook settings')
    print('PyTorch:', torch.__version__)
    print('GPU:', torch.cuda.get_device_name(0))
    candidates = [
        path
        for path in Path('/kaggle/input').rglob('chronopde.h5')
        if path.is_file() and path.stat().st_size == EXPECTED_DATA_SIZE
    ]
    if not candidates:
        raise FileNotFoundError('Attach the private dataset containing chronopde.h5')
    DATA = candidates[0]
    digest = hashlib.sha256()
    with DATA.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    if digest.hexdigest() != EXPECTED_DATA_SHA256:
        raise ValueError('chronopde.h5 checksum does not match the frozen dataset')
    print('Verified dataset:', DATA)

    subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPOSITORY)], check=True)
    subprocess.run(
        ['git', '-C', str(REPOSITORY), 'checkout', '--detach', IMPLEMENTATION_COMMIT],
        check=True,
    )
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--ignore-requires-python', '-e', '.'],
        cwd=REPOSITORY,
        check=True,
    )
    command = [
        sys.executable,
        'scripts/diagnose_loss_alignment.py',
        '--config',
        'configs/project.yaml',
        '--data-path',
        str(DATA),
        '--device',
        'cuda',
        '--max-steps',
        '5000',
        '--evaluation-interval',
        '100',
    ]
    completed = subprocess.run(command, cwd=REPOSITORY, check=False)
    return_code = completed.returncode
    if return_code != 0:
        raise RuntimeError(f'loss-alignment diagnostic returned {return_code}')
except Exception as error:
    failure = {
        'error_type': type(error).__name__,
        'message': str(error),
        'return_code': return_code,
        'traceback': traceback.format_exc(),
    }
    (STATE_ROOT / 'failure.json').write_text(
        json.dumps(failure, indent=2) + '\n', encoding='utf-8'
    )
finally:
    if SOURCE_OUTPUT.is_dir():
        shutil.copytree(SOURCE_OUTPUT, STATE_ROOT / 'loss_alignment', dirs_exist_ok=True)
    package = shutil.make_archive(str(PACKAGE_BASE), 'zip', root_dir=STATE_ROOT)
    print('Diagnostic status:', 'failed' if failure else 'completed')
    print('Package:', package)
    display(FileLink(package))


In [ ]:
summary_path = STATE_ROOT / 'loss_alignment/suite_summary.json'
if summary_path.is_file():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2))
    print('Original gate passed:', summary['passed'])
    print('Next route:', summary['route'])
else:
    print((STATE_ROOT / 'failure.json').read_text())
